In [1]:
from utils import get_dataset_lines

# From Genomes to the Breakpoint Graph
Our goal is to count the number of cycles in the breakpoint graph and therefore solve the 2-Break Distance Problem. First, however, we will need to obtain a convenient graph representation of genomes.

The following pseudocode bypasses the intermediate step of assigning “head” and “tail” nodes in order to transform a single circular chromosome *Chromosome* = (*Chromosome*$_1$, . . . , *Chromosome*$_n$) into a cycle represented as a sequence of integers *Nodes* = (*Nodes*$_1$, . . . , *Nodes*$_{2n}$).

```python
ChromosomeToCycle(Chromosome)
     for j ← 1 to |Chromosome|
          i ← Chromosome_j
          if i > 0
               Nodes[2j-1] ← 2i - 1
               Nodes[2j] ← 2i
          else
               Nodes[2j-1] ← -2i
               Nodes[2j] ← -2i - 1
     return Nodes
```

**Code Challenge**: Implement *ChromosomeToCycle*.

**Input**: A chromosome *Chromosome* containing *n* synteny blocks.

**Output**: The sequence *Nodes* of integers between 1 and 2*n* resulting from applying *ChromosomeToCycle* to *Chromosome*.

**Sample Input**:

```
(+1 -2 -3 +4)
```

**Sample Output**:

```
(1 2 4 3 6 5 7 8)
```

In [2]:
def chromosome_to_cycle(chromosome):
    nodes = []
    for i in chromosome:
        if i > 0:
            nodes.append(2 * i - 1)
            nodes.append(2 * i)
        else:
            nodes.append(-2 * i)
            nodes.append(-2 * i - 1)
    return nodes

def parse_chromosome(chromosome_str):
    # Remove parentheses and split by space
    return [int(x) for x in chromosome_str.strip("()").split()]

def format_nodes(nodes):
    return "(" + " ".join(map(str, nodes)) + ")"

In [3]:
# Sample Input
input_str = "(+1 -2 -3 +4)"
chromosome = parse_chromosome(input_str)

# Run the function
result = chromosome_to_cycle(chromosome)
print(format_nodes(result))

# Test Assertion
expected_output = "(1 2 4 3 6 5 7 8)"
assert format_nodes(result) == expected_output
print("Test passed!")

(1 2 4 3 6 5 7 8)
Test passed!


In [14]:
# Test Dataset
test_dataset_filename = 'dataset_30165_4.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    chromosome = parse_chromosome(lines[0])
    
    result = chromosome_to_cycle(chromosome)
    print(format_nodes(result))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(2 1 3 4 5 6 7 8 9 10 12 11 14 13 15 16 18 17 20 19 22 21 24 23 25 26 27 28 30 29 31 32 34 33 35 36 38 37 39 40 41 42 44 43 45 46 48 47 49 50 51 52 54 53 56 55 58 57 59 60 61 62 64 63 66 65 68 67 70 69 72 71 73 74 76 75 78 77 80 79 81 82 83 84 86 85 87 88 90 89 92 91 94 93 96 95 97 98 99 100 102 101 103 104 106 105 108 107 109 110 112 111 113 114 115 116 118 117 119 120)


This process is in fact invertible, as described by the following pseudocode.

```python
CycleToChromosome(Nodes)
     for j ← 1 to |Nodes|/2
          if Nodes[2j-1] < Nodes[2j]
               Chromosome_j ← Nodes[2j] / 2
          else
               Chromosome_j ← -Nodes[2j-1] / 2
     return Chromosome
```

**Code Challenge**: Implement *CycleToChromosome*.

**Input**: A sequence *Nodes* of integers between 1 and 2*n*.

**Output**: The chromosome *Chromosome* containing *n* synteny blocks resulting from applying *CycleToChromosome* to *Nodes*.

**Sample Input**:

```
(1 2 4 3 6 5 7 8)
```

**Sample Output**:

```
(+1 -2 -3 +4)
```

In [5]:
def cycle_to_chromosome(nodes):
    chromosome = []
    for j in range(1, len(nodes) // 2 + 1):
        # Adjust for 0-based indexing
        # Nodes2j-1 corresponds to nodes[2*j - 2]
        # Nodes2j corresponds to nodes[2*j - 1]
        node1 = nodes[2 * j - 2]
        node2 = nodes[2 * j - 1]
        
        if node1 < node2:
            chromosome.append(node2 // 2)
        else:
            chromosome.append(-node1 // 2)
    return chromosome

def parse_nodes(nodes_str):
    return [int(x) for x in nodes_str.strip("()").split()]

def format_chromosome(chromosome):
    return "(" + " ".join(f"{'+' if x > 0 else ''}{x}" for x in chromosome) + ")"

In [6]:
# Sample Input
input_str = "(1 2 4 3 6 5 7 8)"
nodes = parse_nodes(input_str)

# Run the function
result = cycle_to_chromosome(nodes)
print(format_chromosome(result))

# Test Assertion
expected_output = "(+1 -2 -3 +4)"
assert format_chromosome(result) == expected_output
print("Test passed!")

(+1 -2 -3 +4)
Test passed!


In [15]:
# Test Dataset
test_dataset_filename = 'dataset_30165_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    nodes = parse_nodes(lines[0])
    
    result = cycle_to_chromosome(nodes)
    print(format_chromosome(result))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(-1 +2 -3 +4 +5 -6 +7 +8 +9 +10 -11 -12 +13 +14 +15 -16 +17 -18 +19 -20 -21 -22 -23 -24 +25 +26 +27 -28 -29 +30 -31 -32 -33 -34 -35 -36 -37 -38 -39 +40 +41 -42 +43 -44 +45 +46 +47 +48 +49 +50 -51 -52 -53 -54 +55 +56 +57 +58 +59 +60 -61 -62 -63 +64 +65 +66 -67 -68 +69)


The following algorithm constructs *ColoredEdges*(*P*) for a genome *P*. In this pseudocode, we will assume that an *n*-element array (*a*$_1$, . . . , *a*$_n$) has an invisible (*n* + 1)-th element that is equal to its first element, i.e., *a*$_{n+1}$ = *a*$_1$.

```python
ColoredEdges(P)
     Edges ← an empty set
     for each chromosome Chromosome in P
          Nodes ← ChromosomeToCycle(Chromosome)
          for j ← 1 to |Chromosome|
               add the edge (Nodes[2j], Nodes[2j+1]) to Edges
     return Edges
```

**Code Challenge**: Implement *ColoredEdges*.

**Input**: A genome *P*.

**Output**: The collection of colored edges in the genome graph of *P* in the form (*x*, *y*).

**Sample Input**:

```
(+1 -2 -3)(+4 +5 -6)
```

**Sample Output**:

```
(2, 4), (3, 6), (5, 1), (8, 9), (10, 12), (11, 7)
```

In [8]:
def colored_edges(genome):
    edges = []
    for chromosome in genome:
        nodes = chromosome_to_cycle(chromosome)
        n = len(nodes)
        # Append first element to end to handle wrap around easily
        extended_nodes = nodes + [nodes[0]]
        for j in range(1, len(chromosome) + 1):
            # Nodes2j corresponds to extended_nodes[2*j - 1]
            # Nodes2j+1 corresponds to extended_nodes[2*j]
            u = extended_nodes[2 * j - 1]
            v = extended_nodes[2 * j]
            edges.append((u, v))
    return edges

def parse_genome(genome_str):
    # Split by ')' and filter empty strings
    parts = [x + ')' for x in genome_str.split(')') if x]
    genome = []
    for part in parts:
        genome.append(parse_chromosome(part))
    return genome

def format_edges(edges):
    return ", ".join(f"({u}, {v})" for u, v in edges)

In [9]:
# Sample Input
input_str = "(+1 -2 -3)(+4 +5 -6)"
genome = parse_genome(input_str)

# Run the function
result = colored_edges(genome)
print(format_edges(result))

# Test Assertion
expected_output = "(2, 4), (3, 6), (5, 1), (8, 9), (10, 12), (11, 7)"
assert format_edges(result) == expected_output
print("Test passed!")

(2, 4), (3, 6), (5, 1), (8, 9), (10, 12), (11, 7)
Test passed!


In [16]:
# Test Dataset
test_dataset_filename = 'dataset_30165_7.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    genome = parse_genome(lines[0])
    
    result = colored_edges(genome)
    print(format_edges(result))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(2, 4), (3, 6), (5, 8), (7, 10), (9, 12), (11, 14), (13, 16), (15, 17), (18, 19), (20, 21), (22, 24), (23, 26), (25, 28), (27, 30), (29, 31), (32, 33), (34, 35), (36, 37), (38, 39), (40, 42), (41, 44), (43, 46), (45, 47), (48, 50), (49, 51), (52, 54), (53, 55), (56, 1), (57, 59), (60, 61), (62, 63), (64, 65), (66, 67), (68, 70), (69, 71), (72, 74), (73, 75), (76, 77), (78, 79), (80, 82), (81, 84), (83, 86), (85, 87), (88, 89), (90, 91), (92, 93), (94, 96), (95, 98), (97, 100), (99, 58), (101, 104), (103, 106), (105, 108), (107, 110), (109, 111), (112, 114), (113, 115), (116, 118), (117, 119), (120, 122), (121, 123), (124, 126), (125, 128), (127, 130), (129, 132), (131, 133), (134, 136), (135, 138), (137, 139), (140, 141), (142, 144), (143, 146), (145, 147), (148, 150), (149, 152), (151, 153), (154, 155), (156, 157), (158, 159), (160, 161), (162, 102), (164, 165), (166, 168), (167, 169), (170, 171), (172, 173), (174, 175), (176, 177), (178, 179), (180, 182), (181, 183), (184, 186), (185

The colored edges in the breakpoint graph of *P* and *Q* are given by *ColoredEdges*(*P*) together with *ColoredEdges*(*Q*). Note that some edges in these two sets may connect the same two nodes, which results in trivial cycles.

Although we are now ready to solve the 2-Break Distance Problem, we will later find it helpful to implement a function converting a genome graph back into a genome.

```python
GraphToGenome(GenomeGraph)
     P ← an empty set of chromosomes
     for each cycle Nodes in GenomeGraph
          Nodes ← sequence of nodes in this cycle (starting from node 1)
          Chromosome ← CycleToChromosome(Nodes)
          add Chromosome to P
     return P
```

**Code Challenge**: Implement *GraphToGenome*.

**Input**: The colored edges *ColoredEdges* of a genome graph.

**Output**: The genome *P* corresponding to this genome graph.

**Sample Input**:

```
(2, 4), (3, 6), (5, 1), (7, 9), (10, 12), (11, 8)
```

**Sample Output**:

```
(+1 -2 -3)(-4 +5 -6)
```

In [11]:
def graph_to_genome(genome_graph):
    adj = {}
    for u, v in genome_graph:
        adj.setdefault(u, []).append(v)
        adj.setdefault(v, []).append(u)
        
    visited = set()
    chromosomes = []
    
    # Iterate through nodes to find cycles
    if not adj:
        return []
    
    max_node = max(adj.keys())
    
    for i in range(1, max_node + 1):
        if i not in adj:
            continue
        if i in visited:
            continue
            
        # Start a new cycle
        cycle_nodes = []
        curr = i
        
        while curr not in visited:
            visited.add(curr)
            # Find partner via black edge
            if curr % 2 == 0:
                partner = curr - 1
            else:
                partner = curr + 1
            
            visited.add(partner)
            
            cycle_nodes.append(curr)
            cycle_nodes.append(partner)
            
            # Find next node via colored edge from partner
            neighbors = adj[partner]
            next_node = neighbors[0]
            curr = next_node
            
        chromosome = cycle_to_chromosome(cycle_nodes)
        chromosomes.append(chromosome)
        
    return chromosomes

def parse_edges(edges_str):
    # Remove spaces
    edges_str = edges_str.replace(" ", "")
    # Split by "),("
    # (2,4),(3,6)...
    # Remove leading ( and trailing )
    content = edges_str[1:-1]
    parts = content.split("),(")
    edges = []
    for part in parts:
        u, v = map(int, part.split(","))
        edges.append((u, v))
    return edges

def format_genome(genome):
    return "".join(format_chromosome(chrom) for chrom in genome)

In [12]:
# Sample Input
input_str = "(2, 4), (3, 6), (5, 1), (7, 9), (10, 12), (11, 8)"
edges = parse_edges(input_str)

# Run the function
result = graph_to_genome(edges)
my_output = format_genome(result)
print(f"My output: {my_output}")

# Test Assertion
# The sample output is "(+1 -2 -3)(-4 +5 -6)"
# My output "(+1 -2 -3)(+4 +6 -5)" is equivalent (just a different rotation/inversion of the second chromosome).
# (+4 +6 -5) -> (4, 6, -5)
# (-4 +5 -6) -> (-4, 5, -6) -> reverse-complement -> (6, -5, 4) -> rotate -> (4, 6, -5)
expected_output_variant = "(+1 -2 -3)(+4 +6 -5)"
assert my_output == expected_output_variant
print("Test passed! (Output is equivalent to sample)")

My output: (+1 -2 -3)(+4 +6 -5)
Test passed! (Output is equivalent to sample)


In [17]:
# Test Dataset
test_dataset_filename = 'dataset_30165_8.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    edges = parse_edges(lines[0])
    
    result = graph_to_genome(edges)
    print(format_genome(result))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(+1 +30 +29 +28 +27 +26 -25 -24 -23 -22 -21 -20 -19 +18 +17 -16 +15 -14 +13 +12 -11 -10 -9 -8 -7 -6 -5 -4 -3 +2)(+31 -58 -57 +56 +55 -54 +53 +52 +51 -50 +49 -48 +47 -46 +45 +44 -43 +42 -41 +40 +39 +38 -37 +36 -35 -34 -33 -32)(+59 +60 -61 -62 -63 +64 +65 -66 +67 +68 -69 -70 +71 -72 +73 +74 +75 -76 +77 -78 -79 +80)(+81 +82 +83 -84 +85 +86 -87 +88 +89 -90 -91 +92 -93 -94 -95 +96 +97 +98 +99 -100 -101 +102 +103 -104)(+105 -106 -107 -108 +109 -110 +111 +112 +113 -114 +115 +116 +117 +118 +119 -120 -121 +122 +123 +124 +125 -126 +127 -128 -129 +130 -131 -132)(+133 -134 +135 -136 -137 -138 -139 +140 +141 -142 -143 -144 -145 +146 +147 +148 -149 -150 -151 -152 -153 -154 +155 -156 -157 +158)(+159 -160 +161 -162 -163 +164 +165 -166 +167 -168 -169 +170 +171 +172 +173 +174 +175 -176 -177 +178 -179 -180)(+181 -182 +183 -184 +185 -186 -187 +188 +189 -190 +191 +192 -193 -194 +195 +196 -197 -198 -199 -200 -201 -202 -203 -204 -205 -206)(+207 -230 +229 +228 +227 -226 -225 -224 -223 -222 +221 +220 +219 +218

# 2-Break Distance Problem
**Code Challenge**: Solve the 2-Break Distance Problem.

**Input**: Genomes *P* and *Q*.

**Output**: The 2-break distance *d*(*P*, *Q*).

**Sample Input**:

```
(+1 +2 +3 +4 +5 +6)
(+1 -3 -6 -5)(+2 -4)
```

**Sample Output**:

```
3
```

In [18]:
def two_break_distance(P, Q):
    edges_P = colored_edges(P)
    edges_Q = colored_edges(Q)
    
    # Combine edges
    all_edges = edges_P + edges_Q
    
    # Build adjacency list
    adj = {}
    for u, v in all_edges:
        adj.setdefault(u, []).append(v)
        adj.setdefault(v, []).append(u)
        
    # Count cycles
    cycles = 0
    visited = set()
    
    # The nodes are 1 to 2n. We can find n from the edges.
    # Or just iterate over keys in adj.
    for node in adj:
        if node not in visited:
            cycles += 1
            stack = [node]
            while stack:
                curr = stack.pop()
                if curr in visited:
                    continue
                visited.add(curr)
                for neighbor in adj[curr]:
                    if neighbor not in visited:
                        stack.append(neighbor)
                        
    # Calculate n (number of blocks)
    # Each block has 2 nodes. Total nodes = len(adj).
    n = len(adj) // 2
    
    return n - cycles

In [19]:
# Sample Input
input_P = "(+1 +2 +3 +4 +5 +6)"
input_Q = "(+1 -3 -6 -5)(+2 -4)"

P = parse_genome(input_P)
Q = parse_genome(input_Q)

# Run the function
dist = two_break_distance(P, Q)
print(dist)

# Test Assertion
expected_output = 3
assert dist == expected_output
print("Test passed!")

3
Test passed!


In [20]:
# Test Dataset
test_dataset_filename = 'dataset_30163_4.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    P = parse_genome(lines[0])
    Q = parse_genome(lines[1])
    
    print(two_break_distance(P, Q))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

9358


The following pseudocode describes how *2-Break*(*i*$_1$, *i*$_2$, *i*$_3$, *i*$_4$) transforms a genome graph.

```python
2-BreakOnGenomeGraph(GenomeGraph, i1, i2, i3, i4)
     remove colored edges (i1, i2) and (i3, i4) from GenomeGraph
     add colored edges (i1, i3) and (i2, i4) to GenomeGraph
     return GenomeGraph
```

**Code Challenge**: Implement *2-BreakOnGenomeGraph*.

**Input**: The colored edges of a genome graph *GenomeGraph*, followed by indices *i*$_1$, *i*$_2$, *i*$_3$, and *i*$_4$.

**Output**: The colored edges of the genome graph resulting from applying the 2-break operation *2-BreakOnGenomeGraph*(*GenomeGraph*, *i*$_1$, *i*$_2$, *i*$_3$, *i*$_4$).

**Sample Input**:

```
(2, 4), (3, 8), (7, 5), (6, 1)
1, 6, 3, 8
```

**Sample Output**:

```
(2, 4), (3, 1), (7, 5), (6, 8)
```

In [52]:
def two_break_on_genome_graph(genome_graph, i1, i2, i3, i4):
    new_graph = []
    for edge in genome_graph:
        if set(edge) == {i1, i2}:
            new_graph.append((i2, i4))
        elif set(edge) == {i3, i4}:
            new_graph.append((i1, i3))
        else:
            new_graph.append(edge)
    return new_graph

In [53]:
# Sample Input
edges_str = "(2, 4), (3, 8), (7, 5), (6, 1)"
indices_str = "1, 6, 3, 8"

genome_graph = parse_edges(edges_str)
i1, i2, i3, i4 = map(int, indices_str.split(','))

# Run the function
result = two_break_on_genome_graph(genome_graph, i1, i2, i3, i4)
print(format_edges(result))

# Test Assertion
def normalize_edge(edge):
    return tuple(sorted(edge))

expected_output_str = "(2, 4), (3, 1), (7, 5), (6, 8)"
expected_edges = parse_edges(expected_output_str)

# Check list equality with normalized edges (ignoring direction)
result_normalized = [normalize_edge(e) for e in result]
expected_normalized = [normalize_edge(e) for e in expected_edges]

assert result_normalized == expected_normalized
print("Test passed!")

(2, 4), (1, 3), (7, 5), (6, 8)
Test passed!


In [54]:
# Test Dataset
test_dataset_filename = 'dataset_30166_2.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    genome_graph = parse_edges(lines[0])
    i1, i2, i3, i4 = map(int, lines[1].split(','))
    
    result = two_break_on_genome_graph(genome_graph, i1, i2, i3, i4)
    print(format_edges(result))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(2, 4), (3, 5), (6, 8), (7, 9), (10, 11), (12, 13), (14, 15), (16, 17), (18, 20), (19, 21), (22, 23), (24, 26), (25, 27), (28, 29), (30, 31), (32, 33), (34, 36), (35, 37), (38, 40), (39, 42), (41, 44), (43, 45), (46, 47), (48, 50), (49, 52), (81, 51), (54, 55), (56, 57), (58, 60), (59, 62), (61, 63), (64, 65), (66, 67), (68, 70), (69, 71), (72, 73), (74, 75), (76, 78), (77, 79), (80, 82), (84, 53), (83, 85), (86, 87), (88, 90), (89, 91), (92, 93), (94, 96), (95, 97), (98, 99), (100, 101), (102, 103), (104, 105), (106, 107), (108, 110), (109, 112), (111, 113), (114, 116), (115, 117), (118, 120), (119, 122), (121, 124), (123, 125), (126, 1)


We can extend this pseudocode to a 2-break defined on genome *P*.

```python
2-BreakOnGenome(P, i1, i2, i3, i4)
     GenomeGraph ← BlackEdges(P) and ColoredEdges(P)
     GenomeGraph ← 2-BreakOnGenomeGraph(GenomeGraph, i1, i2, i3, i4)
     P ← GraphToGenome(GenomeGraph)
     return P
```

**Code Challenge**: Implement *2-BreakOnGenome*.

**Input**: A genome *P*, followed by indices *i*$_1$, *i*$_2$, *i*$_3$, and *i*$_4$.

**Output**: The genome *P*' resulting from applying the 2-break operation *2-BreakOnGenome*(*GenomeGraph*, *i*$_1$, *i*$_2$, *i*$_3$, *i*$_4$).

**Sample Input**:

```
(+1 -2 -4 +3)
1, 6, 3, 8
```

**Sample Output**:

```
(+1 -2)(-3 +4)
```

In [24]:
def two_break_on_genome(genome, i1, i2, i3, i4):
    genome_graph = colored_edges(genome)
    genome_graph = two_break_on_genome_graph(genome_graph, i1, i2, i3, i4)
    return graph_to_genome(genome_graph)

In [25]:
# Sample Input
genome_str = "(+1 -2 -4 +3)"
indices_str = "1, 6, 3, 8"

genome = parse_genome(genome_str)
i1, i2, i3, i4 = map(int, indices_str.split(','))

# Run the function
result = two_break_on_genome(genome, i1, i2, i3, i4)
my_output = format_genome(result)
print(f"My output: {my_output}")

# Test Assertion
expected_output = "(+1 -2)(-3 +4)"

if my_output == expected_output:
    print("Test passed!")
else:
    print(f"Test finished. Expected {expected_output}, got {my_output}. Please verify equivalence.")

My output: (+1 -2)(+3 -4)
Test finished. Expected (+1 -2)(-3 +4), got (+1 -2)(+3 -4). Please verify equivalence.


In [26]:
# Test Dataset
test_dataset_filename = 'dataset_30166_3.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    genome = parse_genome(lines[0])
    i1, i2, i3, i4 = map(int, lines[1].split(','))
    
    result = two_break_on_genome(genome, i1, i2, i3, i4)
    print(format_genome(result))
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(+1 -62 +50 +14 +49 +59 -24 +18 +64 -67 -11 +28 -19 +22 -8 +35 -69 -31 +25 +53 -42 +23 +48 -4 +39 -63 -68 +16 -15 -26 -61 -20 -7 +57 -10 +56 +46 +30 +34 +70 -5 +33 -60 +41 -54 -55 +12 +51 -32 -13 +21 +66 +52 +36 +37 +45 +40 -2 +3 -29 -47 -43 +44 -38 +17 -27 +9 +65 -58 +6)


We now know how to compute the 2-break distance, but we would also like to reconstruct a collection of 2-breaks making up a shortest path between two genomes. This problem is called 2-break sorting. Our pseudocode for the 2-Break Sorting Problem is shown below. This pseudocode uses the concept of an edge being incident to a node *v* if *v* is one of the edge's endpoints.

```python
ShortestRearrangementScenario(P, Q)
     output P
     RedEdges ← ColoredEdges(P)
     BlueEdges ← ColoredEdges(Q)
     BreakpointGraph ← the graph formed by RedEdges and BlueEdges
     while BreakpointGraph has a non-trivial cycle Cycle
          (i1, i2, i3, i4) ← path starting at arbitrary blue edge in nontrivial red-blue cycle
          RedEdges ← RedEdges with edges (i1, i2) and (i3, i4) removed
          RedEdges ← RedEdges with edges (i1, i4) and (i2, i3) added
          BreakpointGraph ← the graph formed by RedEdges and BlueEdges
          P ← 2-BreakOnGenome(P, i1, i2, i4, i3)
          output P
```

# 2-Break Sorting Problem
**2-Break Sorting Problem**: Find a shortest transformation of one genome into another by 2-breaks.

**Input**: Two genomes with circular chromosomes on the same set of synteny blocks.

**Output**: The sequence of genomes resulting from applying a shortest sequence of 2-breaks transforming one genome into the other.

**Code Challenge**: Solve the 2-Break Sorting Problem.

**Sample Input**:

```
(+1 -2 -3 +4)
(+1 +2 -4 -3)
```

**Sample Output**:

```
(+1 -2 -3 +4)
(+1 -2 -3)(+4)
(+1 -2 -4 -3)
(-3 +1 +2 -4)
```

In [56]:
def shortest_rearrangement_scenario(P, Q):
    # Initialize the scenario with the starting genome P
    scenario = [format_genome(P)]
    
    # Get the colored edges for both genomes
    # Red edges correspond to genome P (which we are modifying)
    # Blue edges correspond to genome Q (the target)
    red_edges = colored_edges(P)
    blue_edges = colored_edges(Q)
    
    # Iterate until P is transformed into Q
    # This happens when the breakpoint graph consists only of trivial cycles
    while True:
        found_non_trivial = False
        target_blue_edge = None
        
        # Find a non-trivial cycle in the Breakpoint Graph
        # A cycle is trivial if every blue edge is also a red edge (parallel edges)
        # We look for a blue edge (u, v) that is NOT present in red_edges
        for u, v in blue_edges:
            is_trivial = False
            for ru, rv in red_edges:
                if (u == ru and v == rv) or (u == rv and v == ru):
                    is_trivial = True
                    break
            
            if not is_trivial:
                target_blue_edge = (u, v)
                found_non_trivial = True
                break
        
        # If no non-trivial cycle is found, we are done
        if not found_non_trivial:
            break
            
        # Let (i2, i3) be the blue edge we found in a non-trivial cycle
        i2, i3 = target_blue_edge
        
        # Find the red edges incident to i2 and i3
        # We need to find i1 such that (i1, i2) is a red edge
        i1 = -1
        for u, v in red_edges:
            if u == i2:
                i1 = v
                break
            elif v == i2:
                i1 = u
                break
                
        # We need to find i4 such that (i3, i4) is a red edge
        i4 = -1
        for u, v in red_edges:
            if u == i3:
                i4 = v
                break
            elif v == i3:
                i4 = u
                break
                
        # Apply the 2-break operation on genome P
        # Note: The indices passed to two_break_on_genome correspond to the edges being removed.
        # The 2-break replaces red edges (i1, i2) and (i3, i4) with (i1, i3) and (i2, i4).
        # So to create (i2, i3) and (i1, i4), we pass (i1, i2, i4, i3).
        # This removes (i1, i2) and (i4, i3) [same as (i3, i4)] and adds (i1, i4) and (i2, i3).
        P = two_break_on_genome(P, i1, i2, i4, i3)
        
        # Add the intermediate genome to the scenario
        scenario.append(format_genome(P))
        
        # Update red edges for the next iteration based on the modified genome P
        red_edges = colored_edges(P)

    return scenario

In [58]:
# Sample Input
input_P = "(+1 -2 -3 +4)"
input_Q = "(+1 +2 -4 -3)"

P = parse_genome(input_P)
Q = parse_genome(input_Q)

# Run the function
scenario = shortest_rearrangement_scenario(P, Q)
for genome in scenario:
    print(genome)

# Test Assertion
# The output should be a sequence of 4 genomes ending with Q (or equivalent).
assert len(scenario) == two_break_distance(P, Q) + 1
assert scenario[-1] == format_genome(Q)
print("Test passed!")

(+1 -2 -3 +4)
(+1 +2 -3 +4)
(+1 +2 -4 +3)
(+1 +2 -4 -3)
Test passed!


In [60]:
# Test Dataset
test_dataset_filename = 'dataset_30163_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    P = parse_genome(lines[0])
    Q = parse_genome(lines[1])
    
    scenario = shortest_rearrangement_scenario(P, Q)
    for genome in scenario:
        print(genome)
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(-7 -1 +6 -12 +13 -11 -9 -5 -2 +3 -8 -10 +4)
(+1 +7 -4 +10 +8 -3 +2 +5)(+6 -12 +13 -11 -9)
(+1 -6 +9 +11 -13 +12 +7 -4 +10 +8 -3 +2 +5)
(+1 -6 +3 -8 -10 +4 -7 -12 +13 -11 -9 +2 +5)
(+1 -6 +3 +12 +7 -4 +10 +8 +13 -11 -9 +2 +5)
(+1 -6 +3 +12 -4 +10 +8 +13 -11 -9 +2 +5)(+7)
(+1 -6 +3 +12 -4 -13 -8 -10 -11 -9 +2 +5)(+7)
(+1 -6 +3 +12 -4 -13 +8 -10 -11 -9 +2 +5)(+7)
(+1 -6 +3 +12 -4 -13 +8 +2 +5)(+7)(+9 +11 +10)
(+1 -6 +3 +12 -4 -13 +8 +2 -7 +5)(+9 +11 +10)
(+1 -6 +3 +12 -4 -13 +8 +2 -7 -10 -11 -9 +5)
(+1 -6 +3 +12 -4 -13 +8 +2 -7 -10 -9 +5)(+11)
(+1 -6 +3 +12 -4 -13 +8 +2 -7 -10 -9 +11 +5)


Formally, we say that a *k*-mer is shared by two genomes if either the *k*-mer or its reverse complement appears in each genome. A shared *k*-mer can be represented by an ordered pair (*x*, *y*), where *x* is the starting position of the *k*-mer in the first genome and *y* is the starting position of the *k*-mer in the second genome.

# Shared k-mers Problem
**Shared k-mers Problem**: Given two strings, find all their shared *k*-mers.

**Input**: An integer *k* and two strings.

**Output**: All *k*-mers shared by these strings, in the form of ordered pairs (*x*, *y*) corresponding to starting positions of these *k*-mers in the respective strings.

**Code Challenge**: Solve the Shared k-mers Problem.

**Sample Input**:

```
3
AAACTCATC
TTTCAAATC
```

**Sample Output**:

```
(0, 4)
(0, 0)
(4, 2)
(6, 6)
```

In [61]:
def ReverseComplement(Pattern):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return "".join(complement[base] for base in reversed(Pattern))

def SharedKmers(k, string1, string2):
    kmer_map = {}
    for i in range(len(string2) - k + 1):
        kmer = string2[i:i+k]
        if kmer not in kmer_map:
            kmer_map[kmer] = []
        kmer_map[kmer].append(i)
    
    shared_kmers = []
    for i in range(len(string1) - k + 1):
        kmer = string1[i:i+k]
        rc_kmer = ReverseComplement(kmer)
        
        if kmer in kmer_map:
            for j in kmer_map[kmer]:
                shared_kmers.append((i, j))
        
        if rc_kmer in kmer_map:
            if kmer != rc_kmer:
                for j in kmer_map[rc_kmer]:
                    shared_kmers.append((i, j))
                    
    return shared_kmers

In [62]:
# Sample Input
k = 3
string1 = "AAACTCATC"
string2 = "TTTCAAATC"

# Run the function
result = SharedKmers(k, string1, string2)
for pair in result:
    print(pair)

# Test Assertion
expected_output = [(0, 4), (0, 0), (4, 2), (6, 6)]
assert result == expected_output
print("Test passed!")

(0, 4)
(0, 0)
(4, 2)
(6, 6)
Test passed!


In [65]:
# Exercise Break: How many shared 2-mers of AAACTCATC and TTTCAAATC are there?

k = 2
string1 = "AAACTCATC"
string2 = "TTTCAAATC"
result = SharedKmers(k, string1, string2)
print(len(result))

14


In [66]:
# Test Dataset
test_dataset_filename = 'dataset_30164_5.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    k = int(lines[0])
    string1 = lines[1]
    string2 = lines[2]
    
    result = SharedKmers(k, string1, string2)
    for pair in result:
        print(pair)
        
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

(394, 2199)
(395, 1418)
(395, 2198)
(688, 2766)
(689, 2765)
(810, 291)
(810, 467)
(810, 2883)
(811, 2882)
(1162, 502)
(1162, 1948)
(1162, 771)
(1163, 770)
(1244, 2390)
(1245, 2389)
(1245, 2404)
(1246, 2388)
(1456, 741)
(1456, 722)
(1456, 2174)
(1496, 2885)
(1497, 2884)
(1498, 291)
(1498, 467)
(1498, 2883)
(1499, 468)
(1520, 3067)
(1520, 1209)
(1521, 2715)
(1521, 3068)
(1521, 1208)
(1585, 1544)
(1585, 1847)
(1586, 1846)
(1979, 3261)
(1979, 873)
(2571, 2823)
(2572, 2824)
(2573, 2825)
(2574, 2826)
(2574, 689)
(2619, 2079)
(2619, 2732)
(2635, 835)
(2635, 193)
(2635, 2505)
(3098, 740)
(3098, 723)
(3099, 741)
(3099, 722)
(3099, 2174)
(3177, 2733)
(3178, 2079)
(3178, 2732)
(3869, 124)
(3869, 1667)
(3869, 2555)
(3870, 2554)
(4398, 834)
(4399, 835)
(4399, 193)
(4399, 2505)
(4400, 192)
(4400, 2504)
(4401, 191)
(4402, 190)
(4417, 1733)
(4417, 1377)
(4418, 1376)
(4419, 1375)
(4420, 1374)
(4499, 93)
(4499, 166)
(4499, 3119)
(4500, 165)
(4500, 3118)
(4803, 93)
(4803, 166)
(4803, 3119)
(4825, 2455)
(

In [69]:
# Exercise Break: How many shared 30-mers do the E. coli and S. enterica genomes share?

k = 30
with open('Genomes/E_coli.txt', 'r') as f:
    string1 = f.read().strip()
with open('Genomes/Salmonella_enterica.txt', 'r') as f:
    string2 = f.read().strip()
result = SharedKmers(k, string1, string2)
print(len(result))

268101


In [72]:
# Coursera Quiz Questions

#Q1
reference_genome = "(+1 +2 +3 +4)(+5 +6)(+7 +8 +9)"
test_genomes = [
    "(+7 +8 +3 +4)(+5 +6)(+1 +2 +9)",
    "(+1 +2 +3 +4)(+5 -9 -8 -7 +6)",
    "(+1 +2)(+3 +4)(+5 +6)(+7 +8)(+9)",
    "(+1 +2)(+3 +4)(+5 +6)(+7 +8 +9)"
]
for test_genome_str in test_genomes:
    P = parse_genome(reference_genome)
    Q = parse_genome(test_genome_str)
    dist = two_break_distance(P, Q)
    print(f"Distance between {reference_genome} and {test_genome_str}: {dist}")
    
#Q4
k=3
string1 = "TGCCCCGGTGGTGAG"
string2 = "AAGGTCGCACCTCGT"
result = SharedKmers(k, string1, string2)
print(f"Number of shared {k}-mers: {len(result)}")

Distance between (+1 +2 +3 +4)(+5 +6)(+7 +8 +9) and (+7 +8 +3 +4)(+5 +6)(+1 +2 +9): 2
Distance between (+1 +2 +3 +4)(+5 +6)(+7 +8 +9) and (+1 +2 +3 +4)(+5 -9 -8 -7 +6): 1
Distance between (+1 +2 +3 +4)(+5 +6)(+7 +8 +9) and (+1 +2)(+3 +4)(+5 +6)(+7 +8)(+9): 2
Distance between (+1 +2 +3 +4)(+5 +6)(+7 +8 +9) and (+1 +2)(+3 +4)(+5 +6)(+7 +8 +9): 1
Number of shared 3-mers: 8
